In [ ]:
import pandas as pd
from pathlib import Path
import numpy as np
import os
import math
from pathlib import Path
import statsmodels.formula.api as smf
# import pyarrow # just for type-hinting
import pyarrow.parquet as pq
from concurrent.futures import ProcessPoolExecutor

In [65]:
ROOT = Path("/home/hanwenying")
DATADIR = ROOT / "rothman-data/transcriptlength"

In [66]:
tpm = pq.ParquetFile(DATADIR / "tpm-gtex-v11.parquet")
genes = pq.read_table(DATADIR / "tpm-gtex-v11.parquet", columns=['Name']).to_pandas().reset_index()
genes = genes.squeeze()

In [80]:
metadata = pd.read_csv(DATADIR / "metadata_dummy.tsv", sep='\t', dtype=str)

In [68]:
bsize = 100
ngenes = tpm.metadata.num_rows
nchunks = math.ceil(ngenes / bsize)

In [69]:
gene_chunks = np.array_split(genes, nchunks)

/home/hanwenying/.conda/envs/bio/lib/python3.14/site-packages/numpy/_core/fromnumeric.py:54: FutureWarning: 'Series.swapaxes' is deprecated and will be removed in a future version. Please use 'Series.transpose' instead.
  return bound(*args, **kwds)


In [70]:
tpm_iter = tpm.iter_batches(batch_size=bsize)
tpm_chunks = []

In [ ]:
for c in range(nchunks):
	tpm_chunks.append(next(tpm_iter))

In [ ]:
def mp_lmm(tpm_chunk_pq: pyarrow.lib.RecordBatch, genechunk: pd.Series):
	tpmchunk = tpm_chunk_pq.to_pandas().reset_index()
	

	tstatschunk = pd.DataFrame(columns=['GENE', 'TSTATISTIC', 'PVALUE'])

	for i, gene in enumerate(genechunk):
		tpm = tpmchunk[tpmchunk['Name'] == gene]
		y = tpm.drop(columns=['Name','Description']).T.reset_index().rename(columns={'index':'SAMPLE', i:'TPM'})

		dm = pd.merge(y, metadata, on='SAMPLE')
		dm['TPM'] = dm['TPM'].astype(float)
		dm['AGE'] = dm['AGE'].astype(int)

		model = smf.mixedlm("TPM ~ AGE + SEX + TISSUE", dm, groups=dm['SUBJID'])
		res = model.fit()

		t,p = res.tvalues['AGE'], res.pvalues['AGE']
		tstatschunk.loc[len(tstatschunk)] = [gene, t, p]

		print(f"gene:{gene}	t:{t}	p:{p}")
	
	return tstatschunk

In [ ]:
tstatschunk = mp_lmm(tpm_chunks[0], gene_chunks[0])